# FRP++_Ex: Hygrothermal strains and stresses
#### **(by Norbert Blanco)**

This notebook computes the hygrothermal strains and stresses of a composite laminate with UD plies.

Converted from the original MATLAB Live Script (`FRPpp2_Ex0704.m`) to Python. It runs in Jupyter and in Google Colab without any additional setup (only `numpy` and `pandas`, both preinstalled in Colab).

## Initialization

Import the packages used throughout the notebook. The helpers `fmt4` and `fmt_sci` are only used to display the results: `fmt4` writes a number with 4 significant figures (blank space as thousands separator and dot for decimals) and `fmt_sci` writes it in scientific notation with 4 significant figures.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

np.set_printoptions(precision=4, suppress=False)


def fmt4(v, tiny=1e-9):
    """4 significant figures in fixed notation, blank space as thousands separator."""
    v = float(v)
    if abs(v) < tiny:
        return "0"
    decimals = max(0, 3 - int(np.floor(np.log10(abs(v)))))
    return f"{v:,.{decimals}f}".replace(",", " ")


def fmt_sci(v, tiny=1e-15):
    """Scientific notation, 4 significant figures."""
    v = 0.0 if abs(v) < tiny else float(v)
    return f"{v:.3e}"


## Definition of material, LSS and applied loads

#### Definition of material properties

- The material is assumed to be transversely isotropic.
- The plies are assumed to be working under plane-stress condition. So, only 4 elastic constants are required: E11 (MPa), E22 (MPa), nu12, G12 (MPa).
- In addition, the 4 thermal and hygroscopic expansion coefficients are needed: $\alpha_{11}$, $\alpha_{22}$, $\beta_{11}$ and $\beta_{22}$.
- CFRP T300/5208 is defined as material 1. Additional materials can be defined following the same structure `Mat[mat_num] = [E11, E22, nu12, G12, alpha11, alpha22, beta11, beta22]`, or by updating the material properties.

In [ ]:
# Mat[mat_num] = [E11, E22, nu12, G12, alpha11, alpha22, beta11, beta22]
Mat = {
    1: [181000, 10300, 0.28, 7170, 0.02e-6, 22.5e-6, 0.0, 0.5],
}


#### Definition of the Laminate Stacking Sequence

- The LSS is defined through material (previously defined), orientation (in degrees) and thickness (in mm).
- The first layer is assumed to be the one on the bottom, according to the Z direction.
- Define as many layers as required following this structure: `L[layer_num] = [mat_num, orientation, thickness]`.
- If the laminate is symmetric, set `LSS_symm = True` and only define the first half of the LSS. For LSS with odd symmetry, define the central ply with half of the thickness.

In [ ]:
LSS_symm = False

# L[layer_num] = [mat_num, orientation (deg), thickness (mm)]
L = np.array([
    [1,   0, 0.125],
    [1,  45, 0.125],
    [1, -45, 0.125],
    [1,  90, 0.125],
])


#### Hygrothermal conditions

- Curing temperature is defined as Tc and room temperature as Tr, both in ºC.
- The moisture content is defined as hr.

In [ ]:
Tc = 122
Tr = 22
hr = 0.005


#### Applied unit forces (N/mm) and moments (N)

- Unit forces and moments are specified as `NM = [Nxx, Nyy, Nxy, Mxx, Myy, Mxy]`.

In [ ]:
NM = np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0])


## Calculations

First it is necessary to complete the LSS in case a symmetric LSS is defined:

In [ ]:
n_lam = L.shape[0]

if LSS_symm:
    L = np.vstack([L, L[::-1]])

n_lam = L.shape[0]
L


#### Calculation and assembly of stiffness, compliance and transformation matrices

The plane-stress compliance matrix S is calculated for each ply according to the definition for a transversely isotropic material. The plane-stress stiffness matrix Q is obtained by inverting S:

$$
S = \begin{bmatrix}
1/E_{11} & -\nu_{12}/E_{11} & 0 \\
-\nu_{12}/E_{11} & 1/E_{22} & 0 \\
0 & 0 & 1/G_{12}
\end{bmatrix}
$$

In [ ]:
S = np.zeros((3, 3, n_lam))
Q = np.zeros((3, 3, n_lam))

for i in range(n_lam):
    i_mat = int(L[i, 0])
    E11, E22, nu12, G12 = Mat[i_mat][:4]

    S[0, 0, i] = 1 / E11
    S[0, 1, i] = -nu12 / E11
    S[1, 0, i] = S[0, 1, i]
    S[1, 1, i] = 1 / E22
    S[2, 2, i] = 1 / G12

    Q[:, :, i] = np.linalg.inv(S[:, :, i])


The transformation matrix T is calculated for each ply according to the definition. The transformation matrix for strain, `Tg`, is also obtained according to the definition:

$$
[T] = \begin{bmatrix}
\cos^2\theta & \sin^2\theta & 2\cos\theta\sin\theta \\
\sin^2\theta & \cos^2\theta & -2\cos\theta\sin\theta \\
-\cos\theta\sin\theta & \cos\theta\sin\theta & \cos^2\theta - \sin^2\theta
\end{bmatrix}
\qquad
[T_\gamma] = \left([T]^{-1}\right)^T
$$

In [ ]:
T = np.zeros((3, 3, n_lam))
Tg = np.zeros((3, 3, n_lam))

for i in range(n_lam):
    theta = L[i, 1] * np.pi / 180
    m = np.cos(theta)
    n = np.sin(theta)

    T[0, 0, i] = m**2
    T[0, 1, i] = n**2
    T[0, 2, i] = 2 * m * n
    T[1, 0, i] = n**2
    T[1, 1, i] = m**2
    T[1, 2, i] = -2 * m * n
    T[2, 0, i] = -m * n
    T[2, 1, i] = m * n
    T[2, 2, i] = m**2 - n**2

    Tg[:, :, i] = np.linalg.inv(T[:, :, i]).T


The transformed compliance and stiffness matrices are obtained for each ply:

In [ ]:
Sb = np.zeros((3, 3, n_lam))
Qb = np.zeros((3, 3, n_lam))

for i in range(n_lam):
    Sb[:, :, i] = np.linalg.inv(Tg[:, :, i]) @ S[:, :, i] @ T[:, :, i]
    Qb[:, :, i] = np.linalg.inv(Sb[:, :, i])


### Calculation of the laminate's constitutive equation

Before determining matrices A, B and D, it is necessary to determine the total thickness of the laminate and the coordinates of each ply surface from the laminate's mid-plane:

In [ ]:
TH = L[:, 2].sum()
th = TH / 2

Z = np.zeros(n_lam + 1)
Z[0] = -th
running = 0.0
for i in range(n_lam):
    running += L[i, 2]
    Z[i + 1] = running - th

Z


#### Calculation of matrices A, B and D

Matrices A, B and D are calculated taking into account the bottom and top coordinates of the layers and the corresponding stiffness matrices:

$$
A_{jk} = \sum_{i=1}^{n} \bar{Q}_{jk,i} (Z_{i+1} - Z_i) \qquad
B_{jk} = \sum_{i=1}^{n} \bar{Q}_{jk,i} \frac{Z_{i+1}^2 - Z_i^2}{2} \qquad
D_{jk} = \sum_{i=1}^{n} \bar{Q}_{jk,i} \frac{Z_{i+1}^3 - Z_i^3}{3}
$$

In [ ]:
A = np.zeros((3, 3))
B = np.zeros((3, 3))
D = np.zeros((3, 3))

for i in range(n_lam):
    A += Qb[:, :, i] * (Z[i + 1] - Z[i])
    B += Qb[:, :, i] * (Z[i + 1]**2 - Z[i]**2) / 2
    D += Qb[:, :, i] * (Z[i + 1]**3 - Z[i]**3) / 3

ABD = np.block([[A, B], [B, D]])


A (N/mm), B (N), D (N·mm)

In [ ]:
#@title { display-mode: "form" }
labels_abd = ['1', '2', '6']


def show_matrix(name, M, unit):
    print(f"{name} =")
    display(pd.DataFrame([[fmt4(v) for v in row] for row in M], index=labels_abd, columns=labels_abd))
    print(unit)
    print()


show_matrix("A", A, "[N/mm]")
show_matrix("B", B, "[N]")
show_matrix("D", D, "[N·mm]")


#### Calculation of the transformed expansion coefficients

The transformed thermal and hygroscopic expansion coefficients are calculated according to the corresponding definition:

$$
[\alpha_{xyz}] = [T_\gamma]^{-1} [\alpha_{123}] \qquad
[\beta_{xyz}] = [T_\gamma]^{-1} [\beta_{123}]
$$

In [ ]:
alpha_123 = np.zeros((3, n_lam))
beta_123 = np.zeros((3, n_lam))
alpha_xyz = np.zeros((3, n_lam))
beta_xyz = np.zeros((3, n_lam))

for i in range(n_lam):
    i_mat = int(L[i, 0])
    alpha11, alpha22, beta11, beta22 = Mat[i_mat][4:8]

    alpha_123[:, i] = [alpha11, alpha22, 0]
    beta_123[:, i] = [beta11, beta22, 0]
    alpha_xyz[:, i] = np.linalg.inv(Tg[:, :, i]) @ alpha_123[:, i]
    beta_xyz[:, i] = np.linalg.inv(Tg[:, :, i]) @ beta_123[:, i]


#### Calculation of the hygrothermal unit forces and moments

The thermal and hygroscopic unit forces and moments are calculated according to the corresponding definition, where the thermal strain is $\varepsilon^T = \alpha_{xyz}\,\Delta T$ (with $\Delta T = T_r - T_c$) and the hygroscopic strain is $\varepsilon^H = \beta_{xyz}\,h_r$:

$$
N^T = \sum_{k=1}^{N} [\bar{Q}]^k [\varepsilon^T]^k (z_k - z_{k-1}) \qquad
N^H = \sum_{k=1}^{N} [\bar{Q}]^k [\varepsilon^H]^k (z_k - z_{k-1})
$$

$$
M^T = \frac{1}{2}\sum_{k=1}^{N} [\bar{Q}]^k [\varepsilon^T]^k (z_k^2 - z_{k-1}^2) \qquad
M^H = \frac{1}{2}\sum_{k=1}^{N} [\bar{Q}]^k [\varepsilon^H]^k (z_k^2 - z_{k-1}^2)
$$

In [ ]:
DeltaT = Tr - Tc
N_T = np.zeros(3)
N_H = np.zeros(3)
M_T = np.zeros(3)
M_H = np.zeros(3)

for i in range(n_lam):
    N_T += Qb[:, :, i] @ alpha_xyz[:, i] * DeltaT * (Z[i + 1] - Z[i])
    N_H += Qb[:, :, i] @ beta_xyz[:, i] * hr * (Z[i + 1] - Z[i])
    M_T += Qb[:, :, i] @ alpha_xyz[:, i] * DeltaT * (Z[i + 1]**2 - Z[i]**2) / 2
    M_H += Qb[:, :, i] @ beta_xyz[:, i] * hr * (Z[i + 1]**2 - Z[i]**2) / 2


#### Calculation of the laminate's strains and stresses at bottom and top surfaces

The calculation of the mid-plane strains and curvatures is carried out taking into account the total unit forces and moments:

$$
\begin{bmatrix} \varepsilon^0 \\ \kappa \end{bmatrix} =
\begin{bmatrix} A & B \\ B & D \end{bmatrix}^{-1}
\begin{bmatrix} N + N^T + N^H \\ M + M^T + M^H \end{bmatrix} =
\begin{bmatrix} A & B \\ B & D \end{bmatrix}^{-1}
\begin{bmatrix} N^{tot} \\ M^{tot} \end{bmatrix}
$$

In [ ]:
N_tot = NM[0:3] + N_T + N_H
M_tot = NM[3:6] + M_T + M_H
NM_tot = np.concatenate([N_tot, M_tot])

eps_kappa = np.linalg.inv(ABD) @ NM_tot


In [ ]:
#@title { display-mode: "form" }
labels_ek = ['eps_xx', 'eps_yy', 'gamma_xy', 'kappa_xx', 'kappa_yy', 'kappa_xy']
units_ek = ['-', '-', '-', '1/mm', '1/mm', '1/mm']

print("eps_kappa =")
display(pd.DataFrame({'value': [fmt_sci(v) for v in eps_kappa], 'unit': units_ek}, index=labels_ek))


Strains at the bottom and top locations of each ply are calculated taking into account the laminate's mid-plane strains and curvatures and the locations of each ply (columns of each array: bottom, top):

$$
[\varepsilon]^k = [\varepsilon^0] + z^k [\kappa]
$$

Stresses are calculated at each position from the mechanical strain, that is, the total strain minus the thermal and hygroscopic strains, and then rotated to the local axes of the ply:

$$
[\sigma_{xyz}]^k = [\bar{Q}]^k \left( [\varepsilon^0] + z^k [\kappa] - [\varepsilon^T]^k - [\varepsilon^H]^k \right)
\qquad
[\sigma_{123}]^k = [T]^k [\sigma_{xyz}]^k
$$

In [ ]:
eps_tot = np.zeros((3, 2, n_lam))
eps_123 = np.zeros((3, 2, n_lam))
sigma_xyz = np.zeros((3, 2, n_lam))
sigma_123 = np.zeros((3, 2, n_lam))

for i in range(n_lam):
    eps_tot[:, 0, i] = eps_kappa[0:3] + Z[i] * eps_kappa[3:6]
    eps_tot[:, 1, i] = eps_kappa[0:3] + Z[i + 1] * eps_kappa[3:6]
    eps_123[:, :, i] = Tg[:, :, i] @ eps_tot[:, :, i]
    for j in range(2):
        sigma_xyz[:, j, i] = Qb[:, :, i] @ (eps_tot[:, j, i] - alpha_xyz[:, i] * DeltaT - beta_xyz[:, i] * hr)
    sigma_123[:, :, i] = T[:, :, i] @ sigma_xyz[:, :, i]


## Results

The resulting strains and stresses are reported in two tables. Within each table, the layers are listed as they are stacked in the laminate (the last layer on top and the first layer at the bottom). For each layer, the values at its top surface are shown above the values at its bottom surface. The columns on the left correspond to the local axes of the ply (11, 22, 12) and the columns on the right to the global axes (xx, yy, xy). Strains are dimensionless and stresses are in MPa. Values are shown with 4 significant figures.

In [ ]:
#@title { display-mode: "form" }
def result_table(values_local, values_global, formatter):
    """One row per layer and position (last layer on top, top surface above bottom surface).
    Columns: local axes (11, 22, 12) on the left and global axes (xx, yy, xy) on the right.
    values_*[:, j, i] -> component, position (0: bottom, 1: top), layer."""
    index, data = [], []
    for i in reversed(range(n_lam)):
        layer = f"Layer {i + 1} ({L[i, 1]:g}°)"
        for j, pos in ((1, "top"), (0, "bottom")):
            index.append((layer, pos))
            data.append([formatter(v) for v in values_local[:, j, i]] +
                        [formatter(v) for v in values_global[:, j, i]])
    columns = pd.MultiIndex.from_tuples(
        [("Local axes", c) for c in ["11", "22", "12"]] +
        [("Global axes", c) for c in ["xx", "yy", "xy"]])
    return pd.DataFrame(data, index=pd.MultiIndex.from_tuples(index), columns=columns)


display(Markdown("**Strain (-)**"))
display(result_table(eps_123, eps_tot, fmt_sci))

display(Markdown("**Stress (MPa)**"))
display(result_table(sigma_123, sigma_xyz, fmt4))
